# **Pré-processamento de Dados do TCGA da API GDC**
TCGA: The Cancer Genome Atlas  
GDC: Genomic Data Commons

# Importação de Bibliotecas

In [1]:
import json

import pandas as pd
import requests

# Constantes

## API

In [2]:
# URL HTML base da API GDC
GDC_API_ENDPOINT = 'https://api.gdc.cancer.gov'

# Endpoint para verificar status e versão da API
STATUS_ENDPOINT = f'{GDC_API_ENDPOINT}/status'

# Endpoint de projetos relacionados aos programas
PROJECTS_ENDPOINT = f'{GDC_API_ENDPOINT}/projects'

# Endpoint de casos relacionados aos projetos
CASES_ENDPOINT = f'{GDC_API_ENDPOINT}/cases'

# Endpoint de arquivos relacionados aos casos
FILES_ENDPOINT = f'{GDC_API_ENDPOINT}/files'

## Pasta de Dados e Pré-definições

In [3]:
# Caminho da pasta de dados intermediários
INTERIM_DATA_PATH = '../../data/interim/prototyping'

# Lista de projetos de interesse
PROJECTS_LIST = [
    'TCGA-KIRP',
    'TCGA-COAD',
    'TCGA-PAAD'
]

# Função

In [4]:
def gdc_api_request(endpoint, params):
    # Requisita os objetos de interesse ao endpoint
    response = requests.post(
        url=endpoint,
        headers={'Content-Type': 'application/json'},
        json=params
    )

    # Converte o conteúdo da resposta da requisição para JSON
    json_response = json.loads(response.content.decode('utf-8'))

    return json_response['data']['hits']

# Status da API

In [5]:
# Requisita o status e a versão da API ao endpoint
response = requests.get(STATUS_ENDPOINT)

# Imprime o conteúdo da resposta
print(response.content.decode('utf-8'))

{"commit":"7c8ffd0436bb0bb4dafed2d191586309ba6618bf","data_release":"Data Release 41.0 - August 28, 2024","data_release_version":{"major":41,"minor":0,"release_date":"2024-08-28"},"status":"OK","tag":"7.5.1","version":1}



# Projetos

## Obtenção dos Projetos de Interesse

In [6]:
# Campos a serem retornados pela requisição ao endpoint
fields = [
    'disease_type',
    'name',
    'primary_site',
    'summary.case_count'
]
fields = ','.join(fields)

# Filtro usado na requisição ao endpoint
filters = {
    'op': 'in',
    'content': {
        'field': 'project_id',
        'value': PROJECTS_LIST
    }
}

# Parâmetros usados na requisição ao endpoint
params = {
    'fields': fields,
    'filters': filters,
    'size': str(len(PROJECTS_LIST))
}

# Obtém a resposta do endpoint de projetos
projects_response = gdc_api_request(PROJECTS_ENDPOINT, params)

## Criação do DataFrame de Projetos

In [7]:
# Cria um DataFrame de projetos de interesse
df_projects = pd.json_normalize(projects_response) \
    .rename(
        columns={
            'id': 'project_id',
            'disease_type': 'project_disease_type',
            'name': 'project_name',
            'primary_site': 'project_primary_site',
            'summary.case_count': 'project_case_count'
        }
    )

In [8]:
# Armazena o DataFrame em um arquivo CSV
file_name = 'tcga-projects-of-interest.csv'
df_projects.to_csv(f'{INTERIM_DATA_PATH}/{file_name}', index=False)

# Imprime o DataFrame de projetos de interesse
pd.set_option('display.max_colwidth', 500)
df_projects

,project_id,project_primary_site,project_disease_type,project_name,project_case_count
0,TCGA-PAAD,[Pancreas],"[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinomas, Epithelial Neoplasms, NOS]",Pancreatic Adenocarcinoma,185
1,TCGA-KIRP,[Kidney],[Adenomas and Adenocarcinomas],Kidney Renal Papillary Cell Carcinoma,291
2,TCGA-COAD,"[Colon, Rectosigmoid junction]","[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas, Complex Epithelial Neoplasms, Epithelial Neoplasms, NOS]",Colon Adenocarcinoma,461


# Casos

## Obtenção dos Casos de Interesse

In [9]:
# Campos a serem retornados pela requisição ao endpoint
fields = [
    'disease_type',
    'files.created_datetime',
    'files.data_format',
    'files.data_type',
    'files.file_id',
    'files.experimental_strategy',
    'files.updated_datetime',
    'primary_site',
    'project.project_id'
]
fields = ','.join(fields)

# Filtros usados na requisição ao endpoint
filters = {
    'op': 'and',
    'content': [
        {
            'op': 'in',
            'content': {
                'field': 'project.project_id',
                'value': PROJECTS_LIST
            }
        },
        {
            'op': 'in',
            'content': {
                'field': 'files.experimental_strategy',
                'value': ['miRNA-Seq', 'RNA-Seq']
            }
        }
    ]
}

# Parâmetros usados na requisição ao endpoint
params = {
    'fields': fields,
    'filters': filters,
    'size': str(df_projects['project_case_count'].sum())
}

# Obtém a resposta do endpoint de casos
cases_response = gdc_api_request(CASES_ENDPOINT, params)

## Criação de DataFrames de Casos

In [10]:
# Cria o DataFrame de casos de interesse com arquivos associados
df_cases_and_files = pd.json_normalize(cases_response) \
    .rename(
        columns={
            'disease_type': 'case_disease_type',
            'id': 'case_id',
            'primary_site': 'case_primary_site',
            'project.project_id': 'project_id'
        }
    )

# Cria o DataFrame de casos de interesse
df_cases = df_cases_and_files.drop(columns=['files'])

In [11]:
# Imprime o DataFrame de casos de interesse com arquivos associados
df_cases_and_files

,case_id,case_primary_site,case_disease_type,files,project_id
0,733d8b6a-ca9d-4a69-8c9c-1f88733e8b68,Colon,Adenomas and Adenocarcinomas,"[{'data_format': 'BCR SSF XML', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': 'fbe4d9b6-4546-40f0-9401-4bcf72509117', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-21T18:19:54.595818-05:00'}, {'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': 'b97b1554-8e02-485e-b9fd-381b9beba7e5', 'data_type': 'Clinical Supplement', 'created_datetime': '2018-05-21T18:19:54.595818-05:00'}, {'data_format': 'BCR Biotab', '...",TCGA-COAD
1,4f786107-3cf5-4ab3-bba4-f399dee23f0e,Colon,Adenomas and Adenocarcinomas,"[{'data_format': 'CEL', 'updated_datetime': '2020-03-31T14:20:39.076209-05:00', 'file_id': '82fcb2e9-ce0b-43ce-9c14-37c7ac6dc794', 'data_type': 'Raw Intensities', 'experimental_strategy': 'Genotyping Array', 'created_datetime': '2020-03-02T19:37:34.524385-06:00'}, {'data_format': 'BCR XML', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': 'f577cca0-6676-4797-8ce4-7dc07d530c65', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-21T18:19:54.595818-05:00'}, {...",TCGA-COAD
2,ad440651-a2de-4bb1-90da-1e5e8975ab59,Colon,Adenomas and Adenocarcinomas,"[{'data_format': 'VCF', 'updated_datetime': '2024-07-30T14:22:43.114694-05:00', 'file_id': '6bb3a7ba-3427-4a58-b969-5b2602c1088f', 'data_type': 'Annotated Somatic Mutation', 'experimental_strategy': 'WXS', 'created_datetime': '2022-02-04T16:48:52.308897-06:00'}, {'data_format': 'PDF', 'updated_datetime': '2022-11-01T11:48:59.117093-05:00', 'file_id': '384fe6fb-18ca-4530-b0aa-2113a6b89e1f', 'data_type': 'Pathology Report', 'created_datetime': '2022-10-10T06:07:45.051052-05:00'}, {'data_format...",TCGA-COAD
3,13eff2e5-e33a-485f-9ba4-8a7ccb3c7528,Colon,Adenomas and Adenocarcinomas,"[{'data_format': 'BAM', 'updated_datetime': '2023-11-14T15:31:07.587668-06:00', 'file_id': 'd67d5b3c-c880-4f1b-abe3-858eff360cea', 'data_type': 'Aligned Reads', 'experimental_strategy': 'WGS', 'created_datetime': '2023-11-01T15:15:33.411378-05:00'}, {'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': 'b97b1554-8e02-485e-b9fd-381b9beba7e5', 'data_type': 'Clinical Supplement', 'created_datetime': '2018-05-21T18:19:54.595818-05:00'}, {'data_format': ...",TCGA-COAD
4,e394e9ec-7288-4ede-9ef8-41b631d100dd,Colon,Complex Epithelial Neoplasms,"[{'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': 'b97b1554-8e02-485e-b9fd-381b9beba7e5', 'data_type': 'Clinical Supplement', 'created_datetime': '2018-05-21T18:19:54.595818-05:00'}, {'data_format': 'VCF', 'updated_datetime': '2024-07-30T21:31:42.338399-05:00', 'file_id': '453c451a-9ab4-4e5c-aed6-33a48bfb7efa', 'data_type': 'Raw Simple Somatic Mutation', 'experimental_strategy': 'WXS', 'created_datetime': '2022-02-04T15:53:17.130951-06:00'}, {'...",TCGA-COAD
...,...,...,...,...,...
923,f91eb33d-76f0-418c-9962-11a5b31ca1dc,Pancreas,Ductal and Lobular Neoplasms,"[{'data_format': 'VCF', 'updated_datetime': '2024-07-30T03:04:11.223617-05:00', 'file_id': '9f32a053-6de6-440b-9517-a31a915f3991', 'data_type': 'Raw Simple Somatic Mutation', 'experimental_strategy': 'WXS', 'created_datetime': '2022-01-25T21:08:24.455028-06:00'}, {'data_format': 'BCR Biotab', 'updated_datetime': '2018-11-01T15:06:10.843096-05:00', 'file_id': '8cefa9b9-2b1a-4e1a-a5c3-2dd1edda8653', 'data_type': 'Biospecimen Supplement', 'created_datetime': '2018-05-22T01:48:10.832969-05:00'},...",TCGA-PAAD
924,f96ab3fe-bb11-4585-a35e-52d400e55ab7,Pancreas,Ductal and Lobular Neoplasms,"[{'data_format': 'BAM', 'updated_datetime': '2023-07-12T10:47:03.487099-05:00', 'file_id': 'f82de0ae-e060-439b-96f1-0bbd2e98e068', 'data_type': 'Aligned Reads', 'experimental_strategy': 'RNA-Seq', 'created_datetime': '2021-12-13T19:30:32.420969-06:00'}, {'data_format': 'BAM', 'updated_datetime': '2023-07-12T10:1

In [12]:
# Armazena o DataFrame em um arquivo CSV
file_name = 'tcga-cases-of-interest.csv'
df_cases.to_csv(f'{INTERIM_DATA_PATH}/{file_name}', index=False)

# Imprime o DataFrame de casos de interesse
df_cases

,case_id,case_primary_site,case_disease_type,project_id
0,733d8b6a-ca9d-4a69-8c9c-1f88733e8b68,Colon,Adenomas and Adenocarcinomas,TCGA-COAD
1,4f786107-3cf5-4ab3-bba4-f399dee23f0e,Colon,Adenomas and Adenocarcinomas,TCGA-COAD
2,ad440651-a2de-4bb1-90da-1e5e8975ab59,Colon,Adenomas and Adenocarcinomas,TCGA-COAD
3,13eff2e5-e33a-485f-9ba4-8a7ccb3c7528,Colon,Adenomas and Adenocarcinomas,TCGA-COAD
4,e394e9ec-7288-4ede-9ef8-41b631d100dd,Colon,Complex Epithelial Neoplasms,TCGA-COAD
...,...,...,...,...
923,f91eb33d-76f0-418c-9962-11a5b31ca1dc,Pancreas,Ductal and Lobular Neoplasms,TCGA-PAAD
924,f96ab3fe-bb11-4585-a35e-52d400e55ab7,Pancreas,Ductal and Lobular Neoplasms,TCGA-PAAD
925,fe01a390-f45e-4352-a253-af8ef7ee9628,Pancreas,Ductal and Lobular Neoplasms,TCGA-PAAD
926,fe699c36-82ef-45dd-b636-8c18a9ef6c76,Pancreas,Ductal and Lobular Neoplasms,TCGA-PAAD


# Arquivos

## Filtragem dos Arquivos de Interesse

In [13]:
# Explode as listas com dicionários com metadados dos arquivos dos casos
df_cases_and_files = df_cases_and_files.explode('files')

# Filtra os arquivos relacionados às estratégias experimentais miRNA-Seq e RNA-Seq
key = 'experimental_strategy'
df_cases_and_files = df_cases_and_files[
    df_cases_and_files['files'].apply(
        lambda x: (
            key in x and (x[key] == 'miRNA-Seq' or x[key] == 'RNA-Seq')
        )
    )
]

# Explode o conteúdo dos dicionários com metadados dos arquivos em colunas
df_cases_and_files = pd.concat(
    objs=[
        df_cases_and_files.reset_index(drop=True),
        pd.json_normalize(df_cases_and_files['files'])
    ],
    axis='columns'
)

# Cria o DataFrame de arquivos de interesse
df_files = df_cases_and_files \
    .query(
        '(data_type == "Gene Expression Quantification") ' +
        'or (data_type == "Isoform Expression Quantification")'
    ) \
    .drop(
        columns=[
            'case_disease_type',
            'case_primary_site',
            'experimental_strategy',
            'files',
            'project_id'
        ]
    ) \
    .reset_index(drop=True)

In [14]:
# Imprime o DataFrame de arquivos de interesse
df_files

,case_id,data_format,updated_datetime,file_id,data_type,created_datetime
0,733d8b6a-ca9d-4a69-8c9c-1f88733e8b68,TSV,2024-07-30T16:17:03.097090-05:00,daba767c-5d22-4de1-81ba-8ad8f5626465,Gene Expression Quantification,2022-01-05T19:42:59.736874-06:00
1,733d8b6a-ca9d-4a69-8c9c-1f88733e8b68,TXT,2024-07-29T19:49:03.713003-05:00,2554c788-36b4-43c9-ac04-0fb26a1cd37f,Isoform Expression Quantification,2018-03-20T13:21:52.936040-05:00
2,4f786107-3cf5-4ab3-bba4-f399dee23f0e,TXT,2024-07-29T19:49:27.806353-05:00,29d6de65-7221-41b3-bb23-346ae07d5fcf,Isoform Expression Quantification,2018-03-20T13:28:21.292082-05:00
3,4f786107-3cf5-4ab3-bba4-f399dee23f0e,TSV,2024-07-30T16:21:21.965186-05:00,91a95c38-5ab7-4166-b4e7-64454cb3d06c,Gene Expression Quantification,2022-01-05T19:12:17.185895-06:00
4,ad440651-a2de-4bb1-90da-1e5e8975ab59,TXT,2024-07-29T19:46:30.023895-05:00,06fa6f79-8369-4ece-bdc8-7bc76b8700c2,Isoform Expression Quantification,2018-03-20T13:20:49.703268-05:00
...,...,...,...,...,...,...
1999,fe01a390-f45e-4352-a253-af8ef7ee9628,TSV,2024-07-30T14:30:15.613344-05:00,1a5d463c-ace6-4ab6-83f4-407cb2d97db7,Gene Expression Quantification,2021-12-13T19:52:54.502337-06:00
2000,fe699c36-82ef-45dd-b636-8c18a9ef6c76,TXT,2024-07-29T11:18:38.884418-05:00,2f1946f7-98c6-42b2-85e2-9ddc0e574c7b,Isoform Expression Quantification,2018-03-20T06:40:13.982534-05:00
2001,fe699c36-82ef-45dd-b636-8c18a9ef6c76,TSV,2024-07-30T14:34:17.787351-05:00,9bfe5f73-1481-41e3-b8fc-d57b057bae31,Gene Expression Quantification,2021-12-13T19:52:21.436206-06:00
2002,ff8dac97-42ff-4f48-81c4-50315ad8de58,TXT,2024-07-29T11:24:41.128516-05:00,6ea36946-edf5-4b82-9a68-353b3ce55528,Isoform Expression Quantification,2018-03-20T06:38:21.377090-05:00


## Obtenção dos Metadados das Amostras

In [15]:
# Lista com os UUIDs dos arquivos de interesse
files_ids = df_files['file_id'].to_list()

# Campos de interesse para a requisição ao endpoint
fields = [
    'cases.samples.tissue_type',
    'cases.samples.sample_type'
]
fields = ','.join(fields)

# Filtro usado na requisição ao endpoint
filters = {
    'op': 'in',
    'content': {
        'field': 'file_id',
        'value': files_ids
    }
}

# Filtros usados na requisição ao endpoint
filters = {
    'op': 'and',
    'content': [
        {
            'op': 'in',
            'content': {
                'field': 'file_id',
                'value': files_ids
            }
        },
        {
            'op': 'in',
            'content': {
                'field': 'cases.samples.sample_type',
                'value': ['Primary Tumor', 'Solid Tissue Normal']
            }
        }
    ]
}

# Parâmetros usados na requisição ao endpoint
params = {
    'fields': fields,
    'filters': filters,
    'size': str(len(files_ids))
}

# Obtém a resposta do endpoint de arquivos
files_response = gdc_api_request(FILES_ENDPOINT, params)

## Explosão dos Metadados de Amostras

In [16]:
# Cria o DataFrame de amostras dos arquivos
df_files_samples = pd.json_normalize(files_response)

# Explode as listas de dicionários com os metadados das amostras
df_samples = pd.json_normalize(
    pd.json_normalize(
        pd.json_normalize(
            df_files_samples.explode('cases')['cases']
        )['samples']
    )[0]
)

# Concatena os metadados explodidos das amostras aos UUIDs dos arquivos
df_files_samples = pd.concat(
    objs=[df_files_samples, df_samples],
    axis='columns'
)

# Rearranja as colunas do DataFrame
df_files_samples = df_files_samples \
    .rename(columns={'id': 'file_id'}) \
    .drop(columns=['cases'])

In [17]:
# Imprime o DataFrame de amostras dos arquivos
df_files_samples

,file_id,sample_type,tissue_type
0,0149679a-8220-47ba-9060-eda9329bee70,Solid Tissue Normal,Normal
1,8059fb1d-8a5e-4363-a168-52b4007bc340,Primary Tumor,Tumor
2,5ea4d90b-cdc6-4996-9f7a-2ab701846de4,Primary Tumor,Tumor
3,b4c0eaa4-6627-45fe-b8c2-ba23bd263b1f,Primary Tumor,Tumor
4,ea4ee461-c0b4-49b1-819a-aac725493e9e,Primary Tumor,Tumor
...,...,...,...
1991,2f1946f7-98c6-42b2-85e2-9ddc0e574c7b,Primary Tumor,Tumor
1992,9bfe5f73-1481-41e3-b8fc-d57b057bae31,Primary Tumor,Tumor
1993,6ea36946-edf5-4b82-9a68-353b3ce55528,Primary Tumor,Tumor
1994,508e7f55-26a6-4281-88c9-5e35d59f56c2,Primary Tumor,Tumor


## Enriquecimento do DataFrame de Arquivos

In [18]:
# Enriquece o DataFrame de arquivos de interesse
df_files = df_files \
    .merge(
        right=df_files_samples,
        left_on='file_id',
        right_on='file_id',
        how='inner'
    ) \
    [[
        'file_id',
        'data_format',
        'data_type',
        'sample_type',
        'tissue_type',
        'created_datetime',
        'updated_datetime',
        'case_id'
    ]]

In [19]:
# Armazena o DataFrame em um arquivo CSV
file_name = 'tcga-files-of-interest.csv'
df_files.to_csv(f'{INTERIM_DATA_PATH}/{file_name}', index=False)

# Imprime o DataFrame enriquecido de arquivos de interesse
df_files

,file_id,data_format,data_type,sample_type,tissue_type,created_datetime,updated_datetime,case_id
0,daba767c-5d22-4de1-81ba-8ad8f5626465,TSV,Gene Expression Quantification,Primary Tumor,Tumor,2022-01-05T19:42:59.736874-06:00,2024-07-30T16:17:03.097090-05:00,733d8b6a-ca9d-4a69-8c9c-1f88733e8b68
1,2554c788-36b4-43c9-ac04-0fb26a1cd37f,TXT,Isoform Expression Quantification,Primary Tumor,Tumor,2018-03-20T13:21:52.936040-05:00,2024-07-29T19:49:03.713003-05:00,733d8b6a-ca9d-4a69-8c9c-1f88733e8b68
2,29d6de65-7221-41b3-bb23-346ae07d5fcf,TXT,Isoform Expression Quantification,Primary Tumor,Tumor,2018-03-20T13:28:21.292082-05:00,2024-07-29T19:49:27.806353-05:00,4f786107-3cf5-4ab3-bba4-f399dee23f0e
3,91a95c38-5ab7-4166-b4e7-64454cb3d06c,TSV,Gene Expression Quantification,Primary Tumor,Tumor,2022-01-05T19:12:17.185895-06:00,2024-07-30T16:21:21.965186-05:00,4f786107-3cf5-4ab3-bba4-f399dee23f0e
4,06fa6f79-8369-4ece-bdc8-7bc76b8700c2,TXT,Isoform Expression Quantification,Primary Tumor,Tumor,2018-03-20T13:20:49.703268-05:00,2024-07-29T19:46:30.023895-05:00,ad440651-a2de-4bb1-90da-1e5e8975ab59
...,...,...,...,...,...,...,...,...
1991,1a5d463c-ace6-4ab6-83f4-407cb2d97db7,TSV,Gene Expression Quantification,Primary Tumor,Tumor,2021-12-13T19:52:54.502337-06:00,2024-07-30T14:30:15.613344-05:00,fe01a390-f45e-4352-a253-af8ef7ee9628
1992,2f1946f7-98c6-42b2-85e2-9ddc0e574c7b,TXT,Isoform Expression Quantification,Primary Tumor,Tumor,2018-03-20T06:40:13.982534-05:00,2024-07-29T11:18:38.884418-05:00,fe699c36-82ef-45dd-b636-8c18a9ef6c76
1993,9bfe5f73-1481-41e3-b8fc-d57b057bae31,TSV,Gene Expression Quantification,Primary Tumor,Tumor,2021-12-13T19:52:21.436206-06:00,2024-07-30T14:34:17.787351-05:00,fe699c36-82ef-45dd-b636-8c18a9ef6c76
1994,6ea36946-edf5-4b82-9a68-353b3ce55528,TXT,Isoform Expression Quantification,Primary Tumor,Tumor,2018-03-20T06:38:21.377090-05:00,2024-07-29T11:24:41.128516-05:00,ff8dac97-42ff-4f48-81c4-50315ad8de58
